# Notebook #25 — Crypto Multi-Strategy Sweep (Bybit linear)

Sweeps multiple intraday/swing strategies across **all USDT-perpetual symbols** fetched by
[00_data_fetching_bybit.ipynb](00_data_fetching_bybit.ipynb), on two base timeframes (M5 + H1),
to discover strategy/parameter combinations that score well on a *balanced* objective:

- **Win rate ≥ 63%** *or* — and we look at — a profit-aware composite score
  `score = net_avg_R × √(trades_per_month) / (1 + |max_DD| / total_net_R)`.
- **Trades per month ≥ 10** (decent frequency).
- **Profit factor ≥ 1.10** (after fees).
- **Max drawdown** kept reasonable relative to total profit.

## Strategies tested

1. **`trend_pullback`** — H1 (+ optional D1) EMA-trend + M5 pullback to EMA20 + RSI cross gate + reaction candle.
2. **`bb_revert_mid`** — Bollinger outer-band touch + RSI extreme + pin/engulf → TP at BB midline (variable RR).
3. **`rsi_extreme`** — Deep RSI overbought/oversold + pin → TP at EMA20.
4. **`donchian_brkout`** — Breakout of N-bar high/low with HTF trend filter.
5. **`macd_pullback`** — HTF trend + MACD histogram turn + EMA20 pullback.

All entries fill at the next bar's open. Exits are SL, TP, or `max_hold_bars` time stop.

## Fee model

Bybit linear USDT-perp taker fee `~0.055 % per side`, so **round-trip 0.11 %** is deducted in
R-multiples (`fee_R = 0.0011 × entry / |entry − SL|`). This is the same number used in the
exchange's own quote table; tighten by using maker orders / VIP rebates in live trading.

## Outputs (written under `results/_sweep_crypto/`)

- `sweep_all.csv` — every parameter combo evaluated.
- `sweep_winners_wr.csv` — meets *WR ≥ 63 %* **and** *TPM ≥ 10*.
- `sweep_winners_profitable.csv` — profitable + PF ≥ 1.10 + TPM ≥ 10 + WR ≥ 50 %.
- `sweep_best_per_symbol.csv` — top-3 profitable configs per (symbol, timeframe) ranked by composite score.

> The winner-per-symbol configurations selected here are productionised in the
> companion notebooks **`26_top_strategy_per_symbol_bybit.ipynb`** and the per-symbol
> winners under `notebooks/winners/`.


## Section 1 — Imports + strategy library

All strategy + backtest logic lives in [_strategy_lib.py](_strategy_lib.py). It exposes:

- `load_ohlcv(symbol, tf, date_from, date_to)`
- `strategy_<name>(m5, h1, ...)` → DataFrame with `signal`, `sl_price`, optional `tp_price` columns.
- `backtest(df, rr=..., max_hold_bars=..., tp_col=...)` → list of `Trade`.
- `stats(trades, df, rr=...)` → metrics dict.

The numba JIT in the backtest means a full M5 grid for one symbol takes ~30 s.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import sys, json, itertools, time
from pathlib import Path
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:.3f}'.format)

sys.path.insert(0, str(Path.cwd()))
from _strategy_lib import (
    list_symbols, run_strategy, summarise,
    strategy_trend_pullback, strategy_bb_revert_midline,
    strategy_rsi_extreme_reversal, strategy_donchian_breakout,
    strategy_macd_pullback,
)

OUT_DIR = Path('./results/_sweep_crypto')
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATE_FROM = '2023-01-01'

SYMBOLS = [s for s in list_symbols() if s.endswith('USDT')]
print(f'Symbols ({len(SYMBOLS)}): {SYMBOLS}')


Symbols (15): ['ADAUSDT', 'AVAXUSDT', 'BCHUSDT', 'BNBUSDT', 'BTCUSDT', 'DOGEUSDT', 'DOTUSDT', 'ETHUSDT', 'LTCUSDT', 'SHIB1000USDT', 'SOLUSDT', 'XAGUSDT', 'XAUUSDT', 'XLMUSDT', 'XRPUSDT']


## Section 2 — Strategy grids

Each strategy is a dict of lists; the Cartesian product is evaluated for every (symbol, base_tf).
RR values are looped on top for the RR-based strategies; BB-revert + RSI-extreme use a `tp_price`
column so RR is N/A.

In [2]:
TREND_GRID = {
    'rsi_os':       [30, 35], 'rsi_ob': [65, 70],
    'rsi_memory':   [5, 10, 15],
    'wick_ratio':   [0.55], 'pullback_atr': [0.4],
    'session':      [(0, 24)],
    'adx_min':      [0, 18], 'atr_min_mult': [0.0],
    'sl_lookback':  [12], 'sl_buf_atr': [0.10],
    'confirms':     [('f_candle', 'f_ema')],
}
BB_REVERT_GRID = {
    'bb_std':         [2.0, 2.5],
    'rsi_os':         [25, 30], 'rsi_ob': [70, 75],
    'require_pin':    [True],
    'session':        [(0, 24)],
    'adx_max':        [0, 25],
    'sl_atr_mult':    [2.5, 3.0, 3.5, 4.0],
    'sl_method':      ['atr'],
    'trade_with_htf': [False, True],
    'tp_target':      ['mid', 'ema_fast'],
}
RSI_EXTREME_GRID = {
    'rsi_os':     [20, 25], 'rsi_ob': [75, 80],
    'rsi_memory': [3, 5],
    'wick_ratio': [0.55, 0.6],
    'session':    [(0, 24)],
    'adx_max':    [0, 25],
    'sl_lookback':[6, 8],
    'sl_buf_atr': [0.20, 0.30],
    'tp_ema_len': [20],
}
DONCHIAN_GRID = {
    'don_len': [20, 40], 'session': [(0, 24)],
    'adx_min': [18], 'sl_atr_mult': [1.5],
    'sl_lookback': [12], 'sl_buf_atr': [0.10],
    'require_htf_trend': [True],
}
MACD_GRID = {
    'ema_fast': [20], 'pullback_atr': [0.4, 0.6],
    'session': [(0, 24)], 'adx_min': [0, 18],
    'sl_lookback': [12], 'sl_buf_atr': [0.10],
}

STRATS_M5 = [
    ('trend_pullback',  strategy_trend_pullback,       TREND_GRID,        True,  [0.75, 1.0, 1.25, 2.0]),
    ('bb_revert_mid',   strategy_bb_revert_midline,    BB_REVERT_GRID,    False, [None]),
    ('rsi_extreme',     strategy_rsi_extreme_reversal, RSI_EXTREME_GRID,  False, [None]),
    ('donchian_brkout', strategy_donchian_breakout,    DONCHIAN_GRID,     False, [1.0, 1.5, 2.0]),
    ('macd_pullback',   strategy_macd_pullback,        MACD_GRID,         False, [0.75, 1.0, 1.5]),
]
STRATS_H1 = [
    ('trend_pullback',  strategy_trend_pullback,    TREND_GRID,    True, [1.0, 1.5, 2.0, 2.5]),
    ('donchian_brkout', strategy_donchian_breakout, DONCHIAN_GRID, False, [1.5, 2.0]),
]

def grid_product(g):
    keys = list(g)
    return [dict(zip(keys, v)) for v in itertools.product(*[g[k] for k in keys])]

for name, _fn, g, _ud, rr in STRATS_M5 + STRATS_H1:
    print(f'  {name:18}  {len(grid_product(g))} param combos × {len(rr)} RR = {len(grid_product(g))*len(rr)}')


  trend_pullback      24 param combos × 4 RR = 96
  bb_revert_mid       256 param combos × 1 RR = 256
  rsi_extreme         128 param combos × 1 RR = 128
  donchian_brkout     2 param combos × 3 RR = 6
  macd_pullback       4 param combos × 3 RR = 12
  trend_pullback      24 param combos × 4 RR = 96
  donchian_brkout     2 param combos × 2 RR = 4


## Section 3 — Run the sweep

This cell is expensive (~30 min on the full universe). The standalone
[_run_sweep.py](_run_sweep.py) script runs the *same* logic; if results are already cached we
skip the in-notebook run and just load them.

In [3]:
all_csv = OUT_DIR / 'sweep_all.csv'
if all_csv.exists():
    print(f'Loading cached results from {all_csv}')
    df_all = pd.read_csv(all_csv)
else:
    rows = []; t0 = time.time(); n = 0
    for tf in ('M5', 'H1'):
        strats = STRATS_M5 if tf == 'M5' else STRATS_H1
        htf    = 'H1' if tf == 'M5' else 'H4'
        max_hold = 96 if tf == 'M5' else 48
        for sym in SYMBOLS:
            for sname, fn, sgrid, use_daily, rr_grid in strats:
                for params in grid_product(sgrid):
                    for rr in rr_grid:
                        kw = dict(use_daily=use_daily, max_hold_bars=max_hold,
                                  date_from=DATE_FROM)
                        if rr is not None: kw['rr'] = float(rr)
                        try:
                            trades, df = run_strategy(sym, tf, htf, fn, params, **kw)
                        except Exception as exc:
                            print(f'  ✗ {sym} {tf} {sname}: {exc!r}'); continue
                        if df.empty: continue
                        row = summarise(trades, df, sym, sname, params,
                                        rr if rr is not None else 0.0, tf, htf)
                        row['params_json'] = json.dumps(params, default=str)
                        del row['params']
                        rows.append(row); n += 1
                        if n % 100 == 0:
                            print(f'... {n} runs in {time.time()-t0:.1f}s')
    df_all = pd.DataFrame(rows)
    df_all.to_csv(all_csv, index=False)
    print(f'\nSaved {len(df_all)} rows -> {all_csv}')
print(df_all[['symbol','strategy','base_tf','win_rate','trades_per_month','profit_factor','total_net_R']].head())


Loading cached results from results\_sweep_crypto\sweep_all.csv
     symbol        strategy base_tf  win_rate  trades_per_month  profit_factor  total_net_R
0   XLMUSDT        ichimoku      H1    57.317             6.948          2.254       30.689
1   DOTUSDT  trend_pullback      H1    53.125             2.981          2.221       76.602
2  AVAXUSDT  trend_pullback      H1    54.918             2.273          2.365       61.428
3   LTCUSDT  trend_pullback      H1    51.220             3.474          2.108       16.658
4   DOTUSDT  trend_pullback      H1    54.819             3.093          2.151       69.428


## Section 4 — Filter to winners (multi-metric)

We compute three views:

| View | Filter |
| --- | --- |
| `winners_wr` | WR ≥ 63 % AND TPM ≥ 10 (user's hard target) |
| `winners_profitable` | total_net_R > 0 AND PF ≥ 1.10 AND TPM ≥ 10 AND WR ≥ 50 |
| `best_per_symbol` | Top-3 profitable per (symbol, TF) by composite score |

The composite score rewards profitable strategies with high frequency and low relative drawdown.

In [4]:
df_all['params'] = df_all['params_json']
safe_net = np.maximum(df_all['total_net_R'].to_numpy(), 1e-9)
dd_pen   = np.where(df_all['total_net_R'] > 0,
                     1.0 + np.abs(df_all['max_drawdown_R']) / safe_net,
                     np.inf)
df_all['score'] = df_all['net_avg_R'] * np.sqrt(np.maximum(df_all['trades_per_month'], 0)) / dd_pen

winners_wr   = df_all[(df_all['win_rate'] >= 63) & (df_all['trades_per_month'] >= 10)]
winners_prof = df_all[(df_all['total_net_R'] > 0) &
                       (df_all['profit_factor'] >= 1.10) &
                       (df_all['trades_per_month'] >= 10) &
                       (df_all['win_rate'] >= 50)]
best_per_sym = (winners_prof.sort_values('score', ascending=False)
                .groupby(['symbol', 'base_tf'], as_index=False).head(3))

winners_wr.to_csv(OUT_DIR / 'sweep_winners_wr.csv', index=False)
winners_prof.to_csv(OUT_DIR / 'sweep_winners_profitable.csv', index=False)
best_per_sym.to_csv(OUT_DIR / 'sweep_best_per_symbol.csv', index=False)

print(f'WR-targets (WR>=63 & TPM>=10) : {len(winners_wr)}')
print(f'Profitable                    : {len(winners_prof)}')
print(f'Best-per-symbol top-3         : {len(best_per_sym)}')


WR-targets (WR>=63 & TPM>=10) : 116
Profitable                    : 3
Best-per-symbol top-3         : 3


## Section 5 — Inspect top-of-the-list

Sort the two winner sets by composite score and look at the top 20 of each.

In [5]:
cols = ['symbol','strategy','base_tf','rr',
        'trades','win_rate','trades_per_month',
        'profit_factor','total_net_R','max_drawdown_R','score']
print('Top 20 by WR (and TPM>=10):')
print(winners_wr.sort_values(['win_rate','trades_per_month'], ascending=[False, False])[cols].head(20).to_string(index=False))
print('\nTop 20 profitable by composite score:')
print(winners_prof.sort_values('score', ascending=False)[cols].head(20).to_string(index=False))
print('\nBest-per-(symbol, TF):')
print(best_per_sym[cols + ['params']].to_string(index=False))


Top 20 by WR (and TPM>=10):
  symbol      strategy base_tf    rr  trades  win_rate  trades_per_month  profit_factor  total_net_R  max_drawdown_R  score
 BTCUSDT bb_revert_mid      M5 0.000     580    75.690            10.806          1.420      -50.289         -56.972 -0.000
 SOLUSDT bb_revert_mid      M5 0.000     570    75.614            10.619          1.451       10.235         -18.377  0.021
 LTCUSDT bb_revert_mid      M5 0.000     135    74.815            11.664          1.232       -6.600         -12.050 -0.000
 LTCUSDT bb_revert_mid      M5 0.000     135    74.074            11.664          1.352       -4.495         -10.914 -0.000
 ADAUSDT bb_revert_mid      M5 0.000     829    73.945            15.444          1.159      -35.645         -39.310 -0.000
DOGEUSDT bb_revert_mid      M5 0.000     549    73.588            10.228          1.285      -12.062         -18.265 -0.000
 BTCUSDT bb_revert_mid      M5 0.000     585    73.333            10.899          1.362      -66.413    

## Section 6 — Per-symbol summary table

Pick the single best (highest composite score) profitable configuration for each symbol and
show it. This is the seed list for the winner notebooks (`26_…`, `winners/…`).

In [6]:
top1 = (winners_prof.sort_values('score', ascending=False)
         .groupby('symbol', as_index=False).head(1)
         .sort_values('score', ascending=False))
print(f'{len(top1)} of {len(SYMBOLS)} symbols have a profitable strategy meeting the bar.')
print(top1[cols + ['params']].to_string(index=False))
top1.to_csv(OUT_DIR / 'top1_per_symbol.csv', index=False)


2 of 15 symbols have a profitable strategy meeting the bar.
 symbol       strategy base_tf    rr  trades  win_rate  trades_per_month  profit_factor  total_net_R  max_drawdown_R  score                                                                                                                                                                                                                                         params
XLMUSDT sr_zone_bounce      M5 1.500     174    51.149            15.034          1.528        5.671         -15.226  0.034 {"swing_lookback": 8, "zone_atr_band": 0.5, "min_touches": 3, "touch_window": 200, "require_pin": true, "wick_ratio": 0.55, "session": [0, 24], "adx_max": 35, "sl_atr_mult": 2.0, "sl_buf_atr": 0.2, "use_htf_trend": true, "rr_for_tp": 1.2}
SOLUSDT  bb_revert_mid      M5 0.000     570    75.614            10.619          1.451       10.235         -18.377  0.021                                                         {"bb_std": 2.0, "rsi_os": 30, "rsi